# Claude Citations API demo

This notebook shows a minimal example of calling the Claude Citations API from Python using the official `anthropic` SDK.
We attach a short policy memo as a document block, enable citations, and ask Claude to summarize it with references.

## 1. Install dependencies (if needed)

Uncomment the cell below if `anthropic` is not already available in your environment.

In [2]:
# ! pip install --upgrade anthropic

## 2. Load your Anthropic API key

Set the `ANTHROPIC_API_KEY` environment variable before running. The helper below optionally reads from `~/.anthropic-key.txt`; adjust as needed.

In [3]:
open /Users/spangher/.anthropic-usc-key.txt

/Users/spangher/.anthropic-usc-key.txt


In [4]:
import os
from pathlib import Path

if "ANTHROPIC_API_KEY" not in os.environ:
    default_key_path = Path.home() / ".anthropic-usc-key.txt"
    if default_key_path.exists():
        os.environ["ANTHROPIC_API_KEY"] = default_key_path.read_text().strip()
        print("Loaded API key from", default_key_path)
    else:
        raise RuntimeError("Set ANTHROPIC_API_KEY or update the helper to point at your key file.")
else:
    print("Using API key from environment.")

Loaded API key from /Users/spangher/.anthropic-usc-key.txt


## 3. Create a sample briefing and question

In practice you can stream in PDFs or longer knowledge bases. For a quick test we feed a plain-text policy memo as a document block and mark it citation-enabled.

In [6]:
policy_excerpt = 'The Department of Energy (DOE) released the 2024 Transmission Readiness Update in January.\nIt highlights that the U.S. grid needs roughly 110,000 new circuit-miles of high-voltage lines by 2035\nto integrate renewable generation. DOE estimates that every billion dollars invested in transmission\nretrofit corridors unlocks more than 4 GW of additional clean energy capacity for regional markets.\n\nThe update also documents how the Grid Resilience and Innovation Partnership (GRIP) program earmarks\n$10.5 billion to co-fund state, tribal, and utility scale upgrades. Early pilots in Colorado and Arizona\nreported a 37% reduction in congestion costs when reconductoring projects paired with dynamic line\nrating telemetry. DOE expects similar savings for Midwestern projects scheduled to break ground in 2025.'
leading_question = "Summarize DOE's grid readiness plan in three bullet points with citations."

## 4. Call the Claude Citations API

We send a single `user` turn containing the natural-language instructions plus a `document` block.
Setting `citations={'enabled': True}` inside the document signals that Claude should cite this source.

In [10]:
from anthropic import Anthropic

client = Anthropic()
model_name = "claude-sonnet-4-6"

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "You are a regulatory analyst. Use the attached memo to answer the question and include inline citations in parentheses.",
            },
            {
                "type": "document",
                "title": "DOE Transmission Readiness Update (2024)",
                "citations": {
                    "enabled": True
                },
                "source": {
                    "type": "text",
                    "media_type": "text/plain",
                    "data": policy_excerpt.strip(),
                },
            },
            {
                "type": "text",
                "text": leading_question,
            },
        ],
    }
]

response = client.messages.create(
    model=model_name,
    max_tokens=400,
    temperature=0,
    messages=messages,
)
response

Message(id='msg_01KnVp3yvJSU7kGQb7M4WT6c', container=None, content=[TextBlock(citations=None, text="Here is a three-bullet summary of DOE's grid readiness plan:\n\n• ", type='text'), TextBlock(citations=[CitationCharLocation(cited_text='It highlights that the U.S. grid needs roughly 110,000 new circuit-miles of high-voltage lines by 2035\nto integrate renewable generation. ', document_index=0, document_title='DOE Transmission Readiness Update (2024)', end_char_index=229, file_id=None, start_char_index=91, type='char_location')], text='The U.S. grid requires approximately 110,000 new circuit-miles of high-voltage transmission lines by 2035 in order to successfully integrate renewable generation.', type='text'), TextBlock(citations=None, text='\n\n• ', type='text'), TextBlock(citations=[CitationCharLocation(cited_text='The update also documents how the Grid Resilience and Innovation Partnership (GRIP) program earmarks\n$10.5 billion to co-fund state, tribal, and utility scale upgrades. '

## 5. Pretty-print the answer with citation metadata

`TextBlock` objects include a `citations` list when the model references one of the attached sources.
The helper below unwraps each citation so you can see which span of the source was used.

In [12]:
from anthropic.types import (
    TextBlock,
    CitationCharLocation,
    CitationPageLocation,
    CitationContentBlockLocation,
    CitationsSearchResultLocation,
    CitationsWebSearchResultLocation,
)

from textwrap import indent


def describe_citation(citation):
    if isinstance(citation, CitationCharLocation):
        return (
            f"chars {citation.start_char_index}-{citation.end_char_index} "
            f"in '{citation.document_title or f'document #{citation.document_index}'}'"
        )
    if isinstance(citation, CitationPageLocation):
        return (
            f"pages {citation.start_page_number}-{citation.end_page_number} "
            f"in '{citation.document_title or f'document #{citation.document_index}'}'"
        )
    if isinstance(citation, CitationContentBlockLocation):
        return (
            f"blocks {citation.start_block_index}-{citation.end_block_index} "
            f"in '{citation.document_title or f'document #{citation.document_index}'}'"
        )
    if isinstance(citation, CitationsSearchResultLocation):
        return f"search result snippet from {citation.url}"
    if isinstance(citation, CitationsWebSearchResultLocation):
        return f"web search snippet from {citation.url}"
    return "unknown citation type"

for block in response.content:
    if isinstance(block, TextBlock):
        print(block.text)
        if block.citations:
            for citation in block.citations:
                print(
                    indent(
                        f"↳ cites {describe_citation(citation)}\nsnippet: {citation.cited_text.strip()}",
                        prefix="    ",
                    )
                )

Here is a three-bullet summary of DOE's grid readiness plan:

• 
The U.S. grid requires approximately 110,000 new circuit-miles of high-voltage transmission lines by 2035 in order to successfully integrate renewable generation.
    ↳ cites chars 91-229 in 'DOE Transmission Readiness Update (2024)'
    snippet: It highlights that the U.S. grid needs roughly 110,000 new circuit-miles of high-voltage lines by 2035
    to integrate renewable generation.


• 
The Grid Resilience and Innovation Partnership (GRIP) program has earmarked $10.5 billion to co-fund upgrades at the state, tribal, and utility scale.
    ↳ cites chars 396-565 in 'DOE Transmission Readiness Update (2024)'
    snippet: The update also documents how the Grid Resilience and Innovation Partnership (GRIP) program earmarks
    $10.5 billion to co-fund state, tribal, and utility scale upgrades.


• 
Early pilots in Colorado and Arizona demonstrated a 37% reduction in congestion costs through reconductoring projects paired wi

## 6. Optional: tabular view of citations

If you prefer keeping metadata in a dataframe, the snippet below normalizes each citation into a row.

In [13]:
import pandas as pd

rows = []
for block in response.content:
    if isinstance(block, TextBlock) and block.citations:
        for idx, citation in enumerate(block.citations, start=1):
            rows.append(
                {
                    "answer_excerpt": block.text.strip(),
                    "citation_rank": idx,
                    "snippet": citation.cited_text.strip(),
                    "location": describe_citation(citation),
                }
            )

pd.DataFrame(rows)

,answer_excerpt,citation_rank,snippet,location
0,"The U.S. grid requires approximately 110,000 n...",1,It highlights that the U.S. grid needs roughly...,chars 91-229 in 'DOE Transmission Readiness Up...
1,The Grid Resilience and Innovation Partnership...,1,The update also documents how the Grid Resilie...,chars 396-565 in 'DOE Transmission Readiness U...
2,Early pilots in Colorado and Arizona demonstra...,1,Early pilots in Colorado and Arizona\nreported...,chars 565-805 in 'DOE Transmission Readiness U...


---
**Next steps:** Replace the inline `policy_excerpt` with your own files (PDFs or longer text), adjust `model_name` to the Claude release available in your workspace, and enrich the prompt with the reporting format you need.